In [ ]:
from dotenv import load_dotenv
import os
import pandas as pd
from sqlalchemy import create_engine

load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@"
    f"{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("connected")

connected


In [ ]:
pd.read_sql("SELECT * FROM transactions LIMIT 5", engine)

,transaction_id,client_id,agence_id,produit_id,date_transaction,montant_eur,statut,solde_avant,annee,mois
0,TXN000559,CLI0060,1,1,2022-04-19 02:31:00,2050.42,Complete,16415.10,2022,4
1,TXN001154,CLI0057,2,2,2024-06-20 20:51:00,-143.79,Rejete,42890.81,2024,6
2,TXN000764,CLI0015,3,3,2024-08-28 05:03:00,-396.17,Complete,48489.38,2024,8
3,TXN001598,CLI0045,2,2,2024-01-07 08:16:00,225.20,Complete,43962.51,2024,1
4,TXN001873,CLI0034,2,4,2024-08-11 19:52:00,935.32,Complete,17312.83,2024,8


In [ ]:
pd.read_sql("""
SELECT 
    COUNT(*) AS total_transactions,
    SUM(montant_eur) AS total_ca,
    AVG(montant_eur) AS avg_ticket
FROM transactions
""", engine)

,total_transactions,total_ca,avg_ticket
0,2000,-242279.41,-121.139705


In [ ]:
df = pd.read_sql("""
SELECT annee, mois, SUM(montant_eur) AS total
FROM transactions
GROUP BY annee, mois
ORDER BY annee, mois
""", engine)

df

,annee,mois,total
0,2022,1,-11437.50
1,2022,2,-443.90
2,2022,3,7843.81
3,2022,4,5584.87
4,2022,5,-37642.02
5,2022,6,-4314.80
6,2022,7,-11122.35
7,2022,8,-2241.83
8,2022,9,22385.13
9,2022,10,-826.65


In [ ]:
pd.read_sql("""
SELECT agence_id, SUM(montant_eur) AS ca
FROM transactions
GROUP BY agence_id
ORDER BY ca DESC
""", engine)

,agence_id,ca
0,4,-8645.83
1,6,-12514.78
2,1,-20457.66
3,3,-20878.00
4,5,-21980.43
5,2,-44265.80
6,7,-52901.65
7,8,-60635.26


In [ ]:
pd.read_sql("""
SELECT 
    sc.categorie_risque,
    COUNT(*) AS total,
    COUNT(*) FILTER (WHERE t.statut IN ('Rejete','Refusé','Défaut')) AS defauts
FROM transactions t
JOIN clients c ON t.client_id = c.client_id
JOIN segments_client sc ON c.segment_id = sc.segment_id
GROUP BY sc.categorie_risque
""", engine)

,categorie_risque,total,defauts
0,Medium,113,8
1,High,351,24
2,Low,1536,81


In [ ]:
pd.read_sql("""
SELECT client_id, solde_avant
FROM transactions
ORDER BY solde_avant ASC
LIMIT 10
""", engine)

,client_id,solde_avant
0,CLI0082,103.04
1,CLI0003,112.31
2,CLI0016,127.18
3,CLI0101,179.12
4,CLI0149,212.30
5,CLI0075,222.09
6,CLI0072,260.32
7,CLI0133,267.60
8,CLI0117,308.61
9,CLI0118,335.20
